In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, glob, math, textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def place_legend(ax, where='right'):
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return
    if where == 'right':
        # Legend on the right
        ax.legend(handles, labels,
                  loc='upper left', bbox_to_anchor=(1.02, 1.0),
                  borderaxespad=0.0, frameon=False, fontsize=8)
        # leave room on the right for the legend
        ax.figure.tight_layout(rect=[0.0, 0.0, 0.78, 1.0])
    else:
        # Legend under the plot in multiple columns
        ncol = min(4, max(1, int(len(labels) / 2)))
        ax.legend(handles, labels,
                  loc='upper center', bbox_to_anchor=(0.5, -0.18),
                  ncol=ncol, frameon=False, fontsize=8)
        ax.figure.tight_layout()

# -------------------------- Config --------------------------
DATA_GLOBS = [
    '*.csv',        # logs in the same folder as this nothbook
]

# Choose default comparison metric:
# 'acc_test', 'acc_train', 'loss_test', 'loss_train', or 'mean_FgT_loss'
COMPARE_METRIC = 'acc_test'

# Optional pretty-name remapping for legend / tables
LABEL_MAP = {
    # 'SL on CIFAR-10 Alpha=0.1.csv': 'SL (α=0.1)',
    # 'SL + Projected on CIFAR-10 Alpha=0.1.csv': 'SL+Projected (α=0.1)',
}
# ------------------------------------------------------------

print('Globs:', DATA_GLOBS)

def pretty_label(basename: str) -> str:
    return LABEL_MAP.get(basename, basename)

def load_any(path: str) -> pd.DataFrame:
    """
    Load CSV or XLSX into a normalized wide dataframe.
    Adds 'source' (basename) and 'run_label' (pretty label).
    If the file is a long-form forgetting file (contains 'window' and 'FgT_loss'),
    it will be flagged via 'is_long_forgetting' == True.
    """
    ext = os.path.splitext(path)[1].lower()
    if ext == '.csv':
        df = pd.read_csv(path)
    elif ext in ('.xlsx', '.xls'):
        df = pd.read_excel(path)
    else:
        raise ValueError(f'Unsupported file: {path}')

    df = df.copy()
    base = os.path.basename(path)
    df['source'] = base
    df['run_label'] = pretty_label(base)

    # Detect long-form forgetting exports
    if {'window', 'FgT_loss'}.issubset(df.columns):
        df['is_long_forgetting'] = True
        # Ensure round is present and int if possible
        if 'round' in df.columns and df['round'].notna().any():
            df['round'] = df['round'].astype(int, errors='ignore')
        return df

    df['is_long_forgetting'] = False

    # Normalize: ensure expected columns exist (fill with NaN if missing)
    required = [
        'round', 'acc_train', 'acc_test', 'loss_train', 'loss_test',
        'round_time_sec', 'projection', 'top_r', 'alpha', 'num_users',
        'lr', 'seed', 'mean_FgT_loss', 'lr_server', 'lr_client'
    ]
    for col in required:
        if col not in df.columns:
            df[col] = np.nan

    # prefer explicit server/client LRs; fall back to 'lr'
    if df['lr_server'].isna().all() and 'lr' in df:
        df['lr_server'] = df['lr']
    if df['lr_client'].isna().all() and 'lr' in df:
        df['lr_client'] = df['lr']

    if df['round'].notna().any():
        try:
            df['round'] = df['round'].astype(int)
        except Exception:
            pass
    return df

def add_cumulative_time(df: pd.DataFrame) -> pd.DataFrame:
    """If round_time_sec exists, compute cumulative_time per source."""
    df = df.copy()
    if 'round_time_sec' in df.columns and df['round_time_sec'].notna().any():
        df['cumulative_time_sec'] = df.groupby('source', sort=False)['round_time_sec'].cumsum()
    else:
        df['cumulative_time_sec'] = np.nan
    return df

def plot_metric_over_rounds(df: pd.DataFrame, metric: str, outpng: str):
    if metric not in df.columns:
        print(f'[plot] metric "{metric}" not found. Skipping.')
        return
    fig, ax = plt.subplots(figsize=(9.5, 5))
    any_plotted = False
    for src, g in df.groupby('source', sort=False):
        g = g.sort_values('round')
        if g[metric].notna().any():
            ax.plot(g['round'].values, g[metric].values,
                    label=g['run_label'].iloc[0], linewidth=2)
            any_plotted = True
    if not any_plotted:
        plt.close(fig)
        print(f'[plot] No data for metric "{metric}". Skipped.')
        return
    ax.set_xlabel('Round')
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.3)

    # >>> keep legend off the plot area
    place_legend(ax, where='bottom')     # or where='bottom' if you prefer

    fig.savefig(outpng, dpi=150)
    plt.close(fig)
    print('Saved plot:', outpng)
    

def summarize(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    """
    Create a per-run summary for `metric`:
      - final value (last round)
      - best value across rounds (direction-aware)
      - simple sum over rounds (AUC proxy)
      - total time (if available)
      - AUPF for forgetting: sum of positive values across rounds
    Sorts by the direction-aware "best" column.
    """
    if metric not in df.columns:
        raise ValueError(f'Metric "{metric}" not found in dataframe.')

    df = df.dropna(subset=['round']).sort_values('round')
    gby = df.groupby('source', sort=False)

    # final value at last round
    last_idx = gby['round'].idxmax()
    last = df.loc[last_idx, ['source', metric]].rename(columns={metric: f'final_{metric}'})

    # best value (direction-aware)
    if metric in ('loss_test', 'loss_train', 'mean_FgT_loss'):
        best_series = gby[metric].min()   # lower is better
        ascending = True
    else:
        best_series = gby[metric].max()   # higher is better
        ascending = False
    best = best_series.reset_index(name=f'best_{metric}')

    # simple sum over rounds (area proxy)
    auc = gby[metric].sum().reset_index(name=f'sum_{metric}')

    # cumulative time at last round
    if 'cumulative_time_sec' in df.columns and df['cumulative_time_sec'].notna().any():
        tlast = df.loc[last_idx, ['source','cumulative_time_sec']].rename(columns={'cumulative_time_sec':'total_time_sec'})
    else:
        tlast = pd.DataFrame({'source': df['source'].unique(), 'total_time_sec': np.nan})

    out = last.merge(best, on='source').merge(auc, on='source').merge(tlast, on='source')

    # AUPF only for forgetting metric
    if metric == 'mean_FgT_loss':
        aupf = gby.apply(lambda x: np.maximum(x[metric].fillna(0.0).values, 0.0).sum()) \
                  .reset_index(name='AUPF')
        out = out.merge(aupf, on='source', how='left')

    # attach labels
    labels = df[['source','run_label']].drop_duplicates('source')
    out = out.merge(labels, on='source', how='left')

    out = out.sort_values(f'best_{metric}', ascending=ascending)
    cols = ['run_label','source'] + [c for c in out.columns if c not in ('run_label','source')]
    return out[cols]

def plot_forgetting_heatmap(df_long: pd.DataFrame, target_source: str = None, outpng: str = 'forgetting_heatmap.png'):
    """
    Visualize long-form forgetting (window x round) for a single run.
    Requires columns: source, round, window, FgT_loss
    """
    if df_long.empty:
        print('[heatmap] No long-form forgetting files found.')
        return
    if target_source is None:
        target_source = df_long['source'].iloc[0]
    g = df_long[df_long['source'] == target_source].copy()
    if g.empty:
        print('[heatmap] Source not found in long-form data:', target_source)
        return

    # pivot to (window, round) matrix
    try:
        piv = g.pivot_table(index='window', columns='round', values='FgT_loss', aggfunc='mean')
    except Exception as e:
        print('[heatmap] Pivot failed:', e)
        return

    plt.figure(figsize=(9, 5))
    im = plt.imshow(piv.values, aspect='auto', origin='lower')
    plt.colorbar(im, label='FgT_loss')
    plt.xlabel('Round (t)')
    plt.ylabel('Window (t0)')
    lbl = pretty_label(target_source)
    plt.title(f'Forgetting heatmap: {lbl}')
    # annotate axes with actual indices for clarity
    plt.xticks(ticks=np.arange(len(piv.columns)), labels=piv.columns)
    plt.yticks(ticks=np.arange(len(piv.index)), labels=piv.index)
    plt.tight_layout()
    plt.savefig(outpng, dpi=150)
    plt.close()
    print('Saved heatmap:', outpng)

# -------------------------- Load all --------------------------
paths = []
for pat in DATA_GLOBS:
    paths.extend(glob.glob(pat))

if not paths:
    raise SystemExit('No result files found. Update DATA_GLOBS to point to the CSV/XLSX logs.')

dfs = [load_any(p) for p in paths]
df_all = pd.concat(dfs, ignore_index=True)

# split into wide (per-round metrics) and long-form (per-window forgetting)
df_wide = df_all[df_all['is_long_forgetting'] == False].copy()
df_long = df_all[df_all['is_long_forgetting'] == True].copy()

df_wide = df_wide.sort_values(['source', 'round'])
print('Loaded files (wide):')
for p in sorted(set(df_wide['source'])):
    print(' -', p)
if not df_long.empty:
    print('Loaded files (long-form):')
    for p in sorted(set(df_long['source'])):
        print(' -', p)

print('\nColumns (wide):', sorted(df_wide.columns))

# cumulative time for wide data
df_wide = add_cumulative_time(df_wide)

# -------------------------- Plots --------------------------
# main metric plot
plot_metric_over_rounds(df_wide, COMPARE_METRIC, outpng=f'plot_{COMPARE_METRIC}.png')

# forgetting curve
if 'mean_FgT_loss' in df_wide.columns and df_wide['mean_FgT_loss'].notna().any():
    plot_metric_over_rounds(df_wide, 'mean_FgT_loss', outpng='plot_mean_FgT_loss.png')

# loss curves (if present)
if 'loss_test' in df_wide.columns and df_wide['loss_test'].notna().any():
    plot_metric_over_rounds(df_wide, 'loss_test', outpng='plot_loss_test.png')
if 'loss_train' in df_wide.columns and df_wide['loss_train'].notna().any():
    plot_metric_over_rounds(df_wide, 'loss_train', outpng='plot_loss_train.png')

# cumulative time
if 'cumulative_time_sec' in df_wide.columns and df_wide['cumulative_time_sec'].notna().any():
    plt.figure(figsize=(8, 4.5))
    any_plotted = False
    for src, g in df_wide.groupby('source', sort=False):
        g = g.sort_values('round')
        if g['cumulative_time_sec'].notna().any():
            plt.plot(g['round'].values, g['cumulative_time_sec'].values, label=g['run_label'].iloc[0], linewidth=2)
            any_plotted = True
    if any_plotted:
        plt.xlabel('Round')
        plt.ylabel('cumulative_time_sec')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        ax = plt.gca()
        place_legend(ax, where='bottom')   # or 'bottom'
        plt.savefig('plot_cumulative_time.png', dpi=150)
        plt.close()
        print('Saved plot: plot_cumulative_time.png')

# optional heatmap for the first available long-form file
if not df_long.empty:
    plot_forgetting_heatmap(df_long, target_source=None, outpng='forgetting_heatmap.png')

# -------------------------- Summaries --------------------------
def save_and_print_summary(df_w: pd.DataFrame, metric: str, filename: str):
    try:
        summ = summarize(df_w, metric)
    except ValueError as e:
        print(f'[summary] {e}')
        return None
    print(f'\nSummary for "{metric}":')
    print(summ.to_string(index=False))
    summ.to_csv(filename, index=False)
    print('Saved summary:', filename)
    return summ

summary_main = save_and_print_summary(df_wide, COMPARE_METRIC, filename=f'summary_{COMPARE_METRIC}.csv')

# also summarize forgetting
if 'mean_FgT_loss' in df_wide.columns and df_wide['mean_FgT_loss'].notna().any():
    summary_fgt = save_and_print_summary(df_wide, 'mean_FgT_loss', filename='summary_mean_FgT_loss.csv')
else:
    summary_fgt = None

# ------------------------------------------------------------
# Summarize total time comparison
if 'cumulative_time_sec' in df_wide.columns and df_wide['cumulative_time_sec'].notna().any():
    summary_time = save_and_print_summary(df_wide, 'cumulative_time_sec',
                                          filename='summary_cumulative_time_sec.csv')
else:
    summary_time = None

# quick "winner" printout for the chosen metric
if summary_main is not None and not summary_main.empty:
    top = summary_main.iloc[0]
    print(f'\nWinner by best {COMPARE_METRIC}: {top["run_label"]} ({top["source"]})')
    if 'total_time_sec' in summary_main.columns and not pd.isna(top.get('total_time_sec', np.nan)):
        print(f'Total time (winner): {top["total_time_sec"]:.1f} sec')
else:
    print('\nNo summary could be computed for the main metric.')


Globs: ['*.csv']
Loaded files (wide):
 - Centralize Baseline - CIFAR-100.csv
 - FL - CIFAR-100 α=0.5.csv
 - FOT - CIFAR-100 α=0.5.csv
 - SFLV2 + Projected - CIFAR-100 α=0.5.csv
 - SFLV2 - CIFAR-100 α=0.5.csv
 - SL + Projected on CIFAR-100 α=0.5.csv
 - SL - CIFAR-100 α=0.5.csv

Columns (wide): ['acc_test', 'acc_train', 'alpha', 'feature_mem_cap', 'is_long_forgetting', 'loss_test', 'loss_train', 'lr', 'lr_client', 'lr_server', 'mean_FgT_loss', 'num_users', 'projection', 'rebuild_every', 'round', 'round_time_sec', 'run_label', 'seed', 'source', 'top_r', 'warmup_rounds']
Saved plot: plot_acc_test.png
Saved plot: plot_mean_FgT_loss.png
Saved plot: plot_cumulative_time.png

Summary for "acc_test":
                              run_label                                  source  final_acc_test  best_acc_test  sum_acc_test  total_time_sec
    Centralize Baseline - CIFAR-100.csv     Centralize Baseline - CIFAR-100.csv       59.980000      60.120000   9992.560000     1434.089560
SFLV2 + Projected